## 1. Setup

In [ ]:
!pip install --quiet accelerate
!pip install --quiet bitsandbytes
!pip install --quiet peft
!pip install --quiet trl
!pip install --quiet datasets
!pip install --quiet multiprocess
!pip install --quiet llm-fleet


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Obtaining file:///teamspace/studios/this_studio/fleet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-fleet (pyproject.toml) ... done
  Created wh

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from huggingface_hub import hf_hub_download, login

import torch
import numpy as np
import random
import zlib
import base64
import re
import gc
import pickle
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any
from tqdm import tqdm
import json

import multiprocess as mp
import threading
from queue import Queue

In [4]:
device_count = torch.cuda.device_count()
devices = []

for i in range(device_count):
    devices.append(f"cuda:{i}")

In [ ]:
hf_token = '' # Use your Huggingface token

In [6]:
login(token=hf_token)

In [7]:
repo_id = "microsoft/lost_in_conversation"
filename = "lost_in_conversation.json"

tasks_dataset = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")

In [8]:
with open(tasks_dataset) as f:
    data = json.load(f)

In [9]:
coding_tasks = [item for item in data if item.get('task', "") == "code"]
print(coding_tasks[0])

{'task_id': 'sharded-HumanEval/105', 'prompt': '\ndef by_length(arr):\n    """\n    Given an array of integers, sort the integers that are between 1 and 9 inclusive,\n    reverse the resulting array, and then replace each digit by its corresponding name from\n    "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine".\n\n    For example:\n      arr = [2, 1, 1, 4, 5, 8, 2, 3]   \n            -> sort arr -> [1, 1, 2, 2, 3, 4, 5, 8] \n            -> reverse arr -> [8, 5, 4, 3, 2, 2, 1, 1]\n      return ["Eight", "Five", "Four", "Three", "Two", "Two", "One", "One"]\n    \n      If the array is empty, return an empty array:\n      arr = []\n      return []\n    \n      If the array has any strange number ignore it:\n      arr = [1, -1 , 55] \n            -> sort arr -> [-1, 1, 55]\n            -> reverse arr -> [55, 1, -1]\n      return = [\'One\']\n    """\n', 'test': 'def check(candidate):\n\n    # Check some simple cases\n    assert True, "This prints if this assert fails

In [10]:
from transformers import BitsAndBytesConfig

MODEL_NAME = "google/gemma-3-4b-it"

workers = []
dtype = torch.float16

for i in range(device_count):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float32
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map={'': i})
    workers.append((model, tokenizer))

tokenizer = workers[0][1]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [11]:
generation_kwargs = dict(
    max_new_tokens=1024,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

generation_config = GenerationConfig(
    max_new_tokens=1024,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

In [12]:
prompt_content = """
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

Format:
- [Standalone] Make sure that your answer consists of only one Python function at the top level. Do not wrap with a class or split into multiple functions.
"""

system_prompt = {"role": "system", "content": prompt_content}

def prepare_messages(messages):
  return [system_prompt] + messages

In [13]:
def load_test_cases(sample):
  public_test_cases = json.loads(sample["public_test_cases"])  # type: ignore

  if "private_test_cases" in sample:
      try:
          private_test_cases = json.loads(sample["private_test_cases"])  # type: ignore
      except:
          private_test_cases = json.loads(
              pickle.loads(
                  zlib.decompress(
                      base64.b64decode(sample["private_test_cases"].encode("utf-8"))  # type: ignore
                  )
              )
          )  # type: ignore
  else:
      private_test_cases = []

  return json.dumps(
      {
          "inputs": [
              t["input"]
              for t in public_test_cases + private_test_cases
          ],
          "outputs": [
              t["output"]
              for t in public_test_cases + private_test_cases
          ],
          "fn_name": sample["metadata"].get("func_name", None),
      }
  )

In [14]:
# @title
# https://github.com/LiveCodeBench/LiveCodeBench/blob/998c52d394b836f15fff3b9a29866191108ff81b/lcb_runner/evaluation/testing_util.py
#

import ast
import json
import sys
import faulthandler
import platform

# used for debugging to time steps
from datetime import datetime

# to run the solution files we're using a timing based approach
import signal

from io import StringIO

# used for testing the code that reads from input
from unittest.mock import patch, mock_open

# from pyext import RuntimeModule
from types import ModuleType

from enum import Enum
from decimal import Decimal
import time

import multiprocessing

import_string = "from string import *\nfrom re import *\nfrom datetime import *\nfrom collections import *\nfrom heapq import *\nfrom bisect import *\nfrom copy import *\nfrom math import *\nfrom random import *\nfrom statistics import *\nfrom itertools import *\nfrom functools import *\nfrom operator import *\nfrom io import *\nfrom sys import *\nfrom json import *\nfrom builtins import *\nfrom typing import *\nimport string\nimport re\nimport datetime\nimport collections\nimport heapq\nimport bisect\nimport copy\nimport math\nimport random\nimport statistics\nimport itertools\nimport functools\nimport operator\nimport io\nimport sys\nimport json\nsys.setrecursionlimit(50000)\n"


def truncatefn(s, length=300):
    if isinstance(s, str):
        pass
    else:
        s = str(s)
    if len(s) <= length:
        return s

    return s[: length // 2] + "...(truncated) ..." + s[-length // 2 :]


class CODE_TYPE(Enum):
    call_based = 0
    standard_input = 1


# stuff for setting up signal timer
class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    print("timeout occured: alarm went off")
    raise TimeoutException


# used to capture stdout as a list
# from https://stackoverflow.com/a/16571630/6416660
# alternative use redirect_stdout() from contextlib
class Capturing(list):
    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self._stringio = StringIO()
        # Make closing the StringIO a no-op
        self._stringio.close = lambda x: 1
        return self

    def __exit__(self, *args):
        self.append(self._stringio.getvalue())
        del self._stringio  # free up some memory
        sys.stdout = self._stdout


def clean_if_name(code: str) -> str:
    try:
        astree = ast.parse(code)
        last_block = astree.body[-1]
        if isinstance(last_block, ast.If):
            condition = last_block.test
            if ast.unparse(condition).strip() == "__name__ == '__main__'":
                code = (
                    ast.unparse(astree.body[:-1]) + "\n" + ast.unparse(last_block.body)  # type: ignore
                )
    except:
        pass

    return code


def make_function(code: str) -> str:
    try:
        import_stmts = []
        all_other_stmts = []
        astree = ast.parse(code)
        for stmt in astree.body:
            if isinstance(stmt, (ast.Import, ast.ImportFrom)):
                import_stmts.append(stmt)
            else:
                all_other_stmts.append(stmt)

        function_ast = ast.FunctionDef(
            name="wrapped_function",
            args=ast.arguments(
                posonlyargs=[], args=[], kwonlyargs=[], kw_defaults=[], defaults=[]
            ),
            body=all_other_stmts,
            decorator_list=[],
            lineno=-1,
        )
        main_code = (
            import_string
            + "\n"
            + ast.unparse(import_stmts)  # type: ignore
            + "\n"
            + ast.unparse(function_ast)  # type: ignore
        )
        return main_code
    except Exception as e:
        return code


def call_method(method, inputs):

    if isinstance(inputs, list):
        inputs = "\n".join(inputs)

    inputs_line_iterator = iter(inputs.split("\n"))

    # sys.setrecursionlimit(10000)

    # @patch('builtins.input', side_effect=inputs.split("\n"))
    @patch("builtins.open", mock_open(read_data=inputs))
    @patch("sys.stdin", StringIO(inputs))
    @patch("sys.stdin.readline", lambda *args: next(inputs_line_iterator))
    @patch("sys.stdin.readlines", lambda *args: inputs.split("\n"))
    @patch("sys.stdin.read", lambda *args: inputs)
    # @patch('sys.stdout.write', print)
    def _inner_call_method(_method):
        try:
            return _method()
        except SystemExit as e:
            pass
        finally:
            pass

    return _inner_call_method(method)


def get_function(compiled_sol, fn_name: str):  # type: ignore
    try:
        assert hasattr(compiled_sol, fn_name)
        return getattr(compiled_sol, fn_name)
    except Exception as e:
        return


def compile_code(code: str, timeout: int):
    signal.alarm(timeout)
    try:
        tmp_sol = ModuleType("tmp_sol", "")
        try:
            exec(code, tmp_sol.__dict__)
        except Exception as e:
            print(f"Error executing code: {e}")
            # print(f"     executed:\n{code}")
            raise
        if "class Solution" in code:
            # leetcode wraps solutions in `Solution`
            # this is a hack to check if it is leetcode solution or not
            # currently livecodebench only supports LeetCode but
            # else condition allows future extensibility to other platforms
            compiled_sol = tmp_sol.Solution()
        else:
            # do nothing in the other case since function is accesible
            compiled_sol = tmp_sol

        assert compiled_sol is not None
    finally:
        signal.alarm(0)

    return compiled_sol


def convert_line_to_decimals(line: str) -> tuple[bool, list[Decimal]]:
    try:
        decimal_line = [Decimal(elem) for elem in line.split()]
    except:
        return False, []
    return True, decimal_line


def get_stripped_lines(val: str):
    ## you don't want empty lines to add empty list after splitlines!
    val = val.strip()

    return [val_line.strip() for val_line in val.split("\n")]


def grade_call_based(
    code: str, all_inputs: list, all_outputs: list, fn_name: str, timeout: int
):
    # call-based clean up logic
    # need to wrap in try-catch logic after to catch the correct errors, but for now this is fine.
    code = import_string + "\n\n" + code
    compiled_sol = compile_code(code, timeout)

    if compiled_sol is None:
        return

    method = get_function(compiled_sol, fn_name)

    if method is None:
        return

    all_inputs = [
        [json.loads(line) for line in inputs.split("\n")] for inputs in all_inputs
    ]

    all_outputs = [json.loads(output) for output in all_outputs]

    total_execution = 0
    all_results = []
    for idx, (gt_inp, gt_out) in enumerate(zip(all_inputs, all_outputs)):
        signal.alarm(timeout)
        faulthandler.enable()
        try:
            # can lock here so time is useful
            start = time.time()
            prediction = method(*gt_inp)
            total_execution += time.time() - start
            signal.alarm(0)

            # don't penalize model if it produces tuples instead of lists
            # ground truth sequences are not tuples
            if isinstance(prediction, tuple):
                prediction = list(prediction)

            tmp_result = prediction == gt_out

            # handle floating point comparisons

            all_results.append(tmp_result)

            if not tmp_result:
                return all_results, {
                    "output": truncatefn(prediction),
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                    "error_code": -2,
                    "error_message": "Wrong Answer",
                }
        except Exception as e:
            signal.alarm(0)
            if "timeoutexception" in repr(e).lower():
                all_results.append(-3)
                return all_results, {
                    "error": repr(e),
                    "error_code": -3,
                    "error_message": "Time Limit Exceeded",
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                }
            else:
                all_results.append(-4)
                return all_results, {
                    "error": repr(e),
                    "error_code": -4,
                    "error_message": "Runtime Error",
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                }

        finally:
            signal.alarm(0)
            faulthandler.disable()

    return all_results, {"execution time": total_execution}


def grade_stdio(
    code: str,
    all_inputs: list,
    all_outputs: list,
    timeout: int,
):
    ## runtime doesn't interact well with __name__ == '__main__'
    code = clean_if_name(code)

    ## we wrap the given code inside another function
    code = make_function(code)

    compiled_sol = compile_code(code, timeout)
    if compiled_sol is None:
        return

    method = get_function(compiled_sol, "wrapped_function")

    if method is None:
        return

    all_results = []
    total_execution_time = 0
    for idx, (gt_inp, gt_out) in enumerate(zip(all_inputs, all_outputs)):
        signal.alarm(timeout)
        faulthandler.enable()

        signal.alarm(timeout)
        with Capturing() as captured_output:
            try:
                start = time.time()
                call_method(method, gt_inp)
                total_execution_time += time.time() - start
                # reset the alarm
                signal.alarm(0)
            except Exception as e:
                signal.alarm(0)
                if "timeoutexception" in repr(e).lower():
                    all_results.append(-3)
                    return all_results, {
                        "error": repr(e),
                        "error_code": -3,
                        "error_message": "Time Limit Exceeded",
                        "inputs": truncatefn(gt_inp),
                        "expected": truncatefn(gt_out),
                    }
                else:
                    all_results.append(-4)
                    return all_results, {
                        "error": repr(e),
                        "error_code": -4,
                        "error_message": "Runtime Error",
                        "inputs": truncatefn(gt_inp),
                        "expected": truncatefn(gt_out),
                    }

            finally:
                signal.alarm(0)
                faulthandler.disable()

        prediction = captured_output[0]

        stripped_prediction_lines = get_stripped_lines(prediction)
        stripped_gt_out_lines = get_stripped_lines(gt_out)

        ## WA happens in multiple circumstances
        ## so cache the return to make it clean!
        WA_send_args = {
            "output": truncatefn(prediction),
            "inputs": truncatefn(gt_inp),
            "expected": truncatefn(gt_out),
            "error_code": -2,
        }

        if len(stripped_prediction_lines) != len(stripped_gt_out_lines):
            all_results.append(-2)
            WA_send_args["error_message"] = "Wrong answer: mismatched output length"
            return all_results, WA_send_args

        for output_line_idx, (
            stripped_prediction_line,
            stripped_gt_out_line,
        ) in enumerate(zip(stripped_prediction_lines, stripped_gt_out_lines)):
            WA_send_args["error_message"] = (
                f"Wrong answer at {output_line_idx=}: {truncatefn(stripped_prediction_line)} != {truncatefn(stripped_gt_out_line)}"
            )

            ## CASE 1: exact match
            if stripped_prediction_line == stripped_gt_out_line:
                continue

            ## CASE 2: element-wise comparision
            ## if there are floating elements
            ## use `decimal` library for good floating point comparision
            ## otherwise gotcha: np.isclose(50000000000000000, 50000000000000001) = True
            ## note that we should always be able to convert to decimals

            success, decimal_prediction_line = convert_line_to_decimals(
                stripped_prediction_line
            )
            if not success:
                all_results.append(-2)
                return all_results, WA_send_args
            success, decimal_gtout_line = convert_line_to_decimals(stripped_gt_out_line)
            if not success:
                all_results.append(-2)
                return all_results, WA_send_args

            if decimal_prediction_line == decimal_gtout_line:
                continue

            all_results.append(-2)
            return all_results, WA_send_args
        all_results.append(True)

    return all_results, {"execution time": total_execution_time}


def run_test(sample, tests, test=None, debug=False, timeout=6):
    """
    if test(generated_code) is not None it'll try to run the code.
    otherwise it'll just return an input and output pair.
    """
    signal.signal(signal.SIGALRM, timeout_handler)

    # Disable functionalities that can make destructive changes to the test.
    # max memory is set to 4GB
    reliability_guard()

    if debug:
        print(f"start = {datetime.now().time()}")

    try:
        in_outs = json.loads(tests)
    except ValueError as e:
        raise e
        in_outs = None

    if in_outs:
        if in_outs.get("fn_name") is None:
            which_type = CODE_TYPE.standard_input  # Standard input
            method_name = None

        else:
            which_type = CODE_TYPE.call_based  # Call-based
            method_name = in_outs["fn_name"]

    if debug:
        print(f"loaded input_output = {datetime.now().time()}")

    if test is None:
        assert False, "should not happen: test code is none"
        return in_outs, {"error": "no test code provided"}
    elif test is not None:
        results = []
        sol = import_string
        if debug:
            print(f"loading test code = {datetime.now().time()}")

        if which_type == CODE_TYPE.call_based:
            signal.alarm(timeout)
            try:
                results, metadata = grade_call_based(
                    code=test,
                    all_inputs=in_outs["inputs"],
                    all_outputs=in_outs["outputs"],
                    fn_name=method_name,
                    timeout=timeout,
                )
                return results, metadata
            except Exception as e:
                return [-4], {
                    "error_code": -4,
                    "error_message": f"Error during testing: {e}",
                }
            finally:
                signal.alarm(0)
        elif which_type == CODE_TYPE.standard_input:
            # sol
            # if code has if __name__ == "__main__": then remove it

            signal.alarm(timeout)
            try:
                results, metadata = grade_stdio(
                    code=test,
                    all_inputs=in_outs["inputs"],
                    all_outputs=in_outs["outputs"],
                    timeout=timeout,
                )
                return results, metadata
            except Exception as e:
                return [-4], {
                    "error_code": -4,
                    "error_message": f"Error during testing: {e}",
                }
            finally:
                signal.alarm(0)


def reliability_guard(maximum_memory_bytes=None):
    """
    This disables various destructive functions and prevents the generated code
    from interfering with the test (e.g. fork bomb, killing other processes,
    removing filesystem files, etc.)
    WARNING
    This function is NOT a security sandbox. Untrusted code, including, model-
    generated code, should not be blindly executed outside of one. See the
    Codex paper for more information about OpenAI's code sandbox, and proceed
    with caution.
    """

    if maximum_memory_bytes is not None:
        import resource

        resource.setrlimit(
            resource.RLIMIT_AS, (maximum_memory_bytes, maximum_memory_bytes)
        )
        resource.setrlimit(
            resource.RLIMIT_DATA, (maximum_memory_bytes, maximum_memory_bytes)
        )
        if not platform.uname().system == "Darwin":
            resource.setrlimit(
                resource.RLIMIT_STACK, (maximum_memory_bytes, maximum_memory_bytes)
            )

    faulthandler.disable()

    import builtins

    # builtins.exit = None
    builtins.quit = None

    import os

    os.environ["OMP_NUM_THREADS"] = "1"

    os.kill = None
    os.system = None
    os.putenv = None
    os.remove = None
    os.removedirs = None
    os.rmdir = None
    os.fchdir = None
    os.setuid = None
    os.fork = None
    os.forkpty = None
    os.killpg = None
    os.rename = None
    os.renames = None
    os.truncate = None
    os.replace = None
    os.unlink = None
    os.fchmod = None
    os.fchown = None
    os.chmod = None
    os.chown = None
    os.chroot = None
    os.fchdir = None
    os.lchflags = None
    os.lchmod = None
    os.lchown = None
    os.getcwd = None
    os.chdir = None

    import shutil

    shutil.rmtree = None
    shutil.move = None
    shutil.chown = None

    import subprocess

    subprocess.Popen = None  # type: ignore

    builtins.help = None

    import sys

    sys.modules["ipdb"] = None
    sys.modules["joblib"] = None
    sys.modules["resource"] = None
    sys.modules["psutil"] = None
    sys.modules["tkinter"] = None


def _temp_run(sample, generation, tests, debug, result, metadata_list, timeout):
    res, metadata = run_test(sample, tests, test=generation, debug=debug, timeout=timeout)
    result.append(res)
    metadata_list.append(metadata)


def check_correctness(sample, generation, tests, timeout, debug=False):
    """Check correctness of code generation with a global timeout.
    The global timeout is to catch some extreme/rare cases not handled by the timeouts
    inside `run_test`"""

    manager = multiprocessing.Manager()
    result = manager.list()
    metadata_list = manager.list()
    p = multiprocessing.Process(
        target=_temp_run,
        args=(sample, generation, tests, debug, result, metadata_list, timeout),
    )

    p.start()
    p.join(
        timeout=(timeout + 1) * len(json.loads(tests)["inputs"]) + 5
    )

    if p.is_alive():
        p.kill()

    if not result:
        in_outs = json.loads(tests)
        # consider that all tests failed
        result = [[-1 for i in range(len(in_outs["inputs"]))]]
        if debug:
            print("global timeout")

    return result[0], metadata_list[0], result, metadata_list



In [15]:
def extract_code(text):
    pattern = r".*```python\n(.*?)\n```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches is None or len(matches) == 0:
        return ""
    initial_string = matches[-1] + "\n"

    return initial_string

In [16]:
def evaluate_task(task, conversation, verbose=True):
  final_answer = extract_code(conversation[-1]["content"])
  pred_python_code = final_answer.replace("```python", "").replace("```", "")

  if verbose:
      print(final_answer)
      print(pred_python_code)

  if "def " not in pred_python_code:
    if verbose:
      print("No def found in the last code snippet.")
    return {
      "correct": False,
      "pass@1": 0,
      "score": 0,
      "response": "No def found in the last code snippet."
    }

  # Adding imports for HE-derived samples
  if "prompt" in task:
    # Extract imports from sample["prompt"] -- this affects full
    prompt_ast = ast.parse(task["prompt"])
    imports = []
    for node in prompt_ast.body:
      if isinstance(node, (ast.Import, ast.ImportFrom)):
        imports.append(ast.unparse(node))

    # Prepend imports to pred_python_func
    if imports:
      pred_python_code = "\n".join(imports) + "\n\n" + pred_python_code

  # Force update the function name with the true function name
  old_func_name = pred_python_code.split("def ")[1].split("(")[0].strip()
  pred_python_code = pred_python_code.replace(old_func_name, task["metadata"]["func_name"])

  # load tests
  testcases = load_test_cases(task)

  output, metadata, full_output, full_metadata = check_correctness(task, pred_python_code, testcases, timeout=6)

  if verbose:
    print(full_output)
    print(full_metadata)

    for i, (output, metadata) in enumerate(zip(full_output, full_metadata)):
      print(f"{i}.")
      print(f"Expected: {metadata.get('expected', '')}")
      print("\n")
      print(f"Actual: {metadata.get('output', '')}")
      print("\n")

  all_test_cases_passed = all(o is True for o in output)

  score = len([o for o in output if o is True]) / len(output)
  return {
    "correct": all_test_cases_passed,
    "pass@1": 1 if all_test_cases_passed else 0,
    "score": score,
    "response": full_metadata
  }

In [17]:
mp.set_start_method('spawn', force=True)

def update_pbar_stats(pbar, finished_states):
    correct_count = sum(1 for s in finished_states if s.stats['correct'])
    total = len(finished_states)
    if total == 0: return

    scores = [s.stats['score'] for s in finished_states]

    pbar.set_postfix({
        'acc': f"{correct_count / total:.2f}",
        'avg_score': f"{sum(scores) / total:.2f}"
    })

def compile_statistics(finished_states):
    correct = sum(1 for s in finished_states if s.stats['correct'])
    total = len(finished_states)
    scores = [s.stats['score'] for s in finished_states]

    return {
        'accuracy': correct / total if total else 0,
        'avg score': sum(scores) / total if total else 0
    }, [s.conversation for s in finished_states]

## 2. Connecting transformers to fleet

In [18]:
from fleet import FleetWorker, VectorDSU, Node, ResidualCollection
from fleet.prior_tree import AgglomerativePriorTreeBuilder
from copy import copy

In [19]:
def remove_all_module_hooks(model: torch.nn.Module) -> None:
    """Clears internal hook dictionaries for all modules."""
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()

For the simpler models the process is also straightforward:
* Add a logits processor that will simply apply penalty to the scores
* Get a hook on the target layer and pass the outputs to the worker
* Intercept embeddings to figure out the tokens sampled (not necessary if you use greedy decoding - logits processor can do that then)
* Generate > Evaluate > Backpropagate the reward > Repeat

In [20]:
from transformers import LogitsProcessor, LogitsProcessorList
import traceback

class DeltaInterventionLogitsProcessor(LogitsProcessor):
    def __init__(self, worker):
        super().__init__()
        self.worker = worker

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.worker.hit_threshold:
            delta, _ = self.worker.process_logits(scores.flatten())
            scores = scores - delta

        return scores

def get_layer_n_hook(model, worker):
    def hook_fn(module, input, output):
        hidden_states = output[0] if isinstance(output, tuple) else output
        last_token_hidden_state = hidden_states[:, -1:, :].clone()
        with torch.no_grad():
            normed_state = model.model.language_model.norm(last_token_hidden_state)
            layer_n_logits = model.lm_head(normed_state)

        _ = worker.update_logits(last_token_hidden_state.reshape(-1), layer_n_logits.reshape(1, -1))
        return None
    return hook_fn

def get_embedding_interception_hook(worker):
    def embedding_interception_hook(module, args):
        # args is a tuple of the positional arguments passed to the layer.
        # For embed_tokens, args[0] is the input_ids tensor.
        input_ids = args[0].flatten().tolist()
        if len(input_ids) == 1:
            worker.update_tokens(input_ids[-1])

        return None
    return embedding_interception_hook

def entropy_search_based_sampler(
    model, tokenizer, worker, messages, verbose=True, generation_kwargs=None
):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, return_tensors='pt', add_generation_prompt=True
    )
    prompt_tokens = inputs['input_ids'][0].tolist()
    worker.update_prompt(prompt_tokens)

    intercept_layer = model.model.language_model.layers[worker.layer]

    if generation_kwargs is None:
        generation_kwargs = dict(max_new_tokens=1024, do_sample=False)

    hs_hook_handle = intercept_layer.register_forward_hook(get_layer_n_hook(model, worker))
    token_hook_handle = model.model.language_model.embed_tokens.register_forward_pre_hook(get_embedding_interception_hook(worker))

    my_processors = LogitsProcessorList([
        DeltaInterventionLogitsProcessor(worker)
    ])

    input_ids=inputs["input_ids"].to(model.device)
    attention_mask=inputs["attention_mask"].to(model.device)
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        logits_processor=my_processors,
        **generation_kwargs
    )
    hs_hook_handle.remove()
    token_hook_handle.remove()

    result = {
        'changes': worker.queue,
        'completion': tokenizer.decode(outputs[0][input_ids.shape[1]:].cpu().detach())
    }

    return result

def worker_closure(model, tokenizer, task, rank, context, iterations, queue, verbose, return_trajectory):
    results = []
    logit_dim = model.config.text_config.vocab_size

    worker = FleetWorker(
        rank, context['dsu'], context['root'], -2, context['threshold'], logit_dim,
        prior_tree= context['prior_tree'], verbose=False, use_reward_penalty=True, return_trajectory=return_trajectory
    )

    for i in range(iterations):
        try:
            result = entropy_search_based_sampler(
                model, tokenizer, worker, task.conversation, verbose=verbose
            )

            solution = result['completion']
            extended_convo = [message for message in task.conversation]
            extended_convo.append({'role': 'assistant', 'content': solution})

            try:
                stats = evaluate_task(task.task_data, extended_convo, verbose=False)
            except:
                stats = {'correct': False, 'score': 0.0}

            result['stats'] = stats
            score = stats['score']

            trajectory = worker.finish_iteration(score)
            if trajectory is not None:
                result['trajectory'] = trajectory

            result['entropy_data'] = (copy(worker.entropies), copy(worker.varentropies))
            results.append(result)
        except Exception:
            results.append(traceback.format_exc())
            remove_all_module_hooks(model.model.language_model)

    queue.put(results)

## 3. Benchmarking the model

In [21]:
@dataclass
class TaskState:
    task_id: int
    task_data: Any
    conversation: List[Dict[str, str]] = field(default_factory=list)
    attempts: int = 0
    stats: Dict = field(default_factory=dict)

@dataclass
class BenchmarkResults:
    items: List[TaskState]
    metadata: Dict[str, Any]

In [29]:
def benchmark(
    tasks_source,
    workers,
    metadata,
    verbose=False,
    return_trajectory=True,
    use_prior_tree=False,
    max_tasks=None,
    max_attempts=2,
    batch_size=8
):
    if max_tasks is not None and max_tasks < len(tasks_source):
        tasks_source = random.sample(tasks_source, max_tasks)

    tasks = []
    for i, task in enumerate(tasks_source):
        conversation = [system_prompt]

        content = ""
        if content == "":
            content = task.get('prompt', "").strip()

        if content == "":
            content = task.get('question_content', "").strip()

        conversation.append({'role': 'user', 'content': content})

        task_state = TaskState(task_id=i, task_data=task, conversation=conversation)
        tasks.append(task_state)

    progress_bar = tqdm(total=len(tasks_source))
    finished_states = []
    residuals = []

    traces = []
    tree_builder = AgglomerativePriorTreeBuilder()

    def merge_stats(stats, other_stats):
        return {
            'correct': stats['correct'] or other_stats['correct'],
            'score': max(stats['score'], other_stats['score']),
        }

    for task in tasks:
        processes = []
        solutions = []

        best_solution = None
        dsu = VectorDSU()
        residual_collecton = ResidualCollection(dsu=dsu)

        root = dsu.node_store[dsu.create_node()]
        stats = {
            'correct': False,
            'score': 0.0,
        }

        queue = Queue()

        try:
            context = {
                'root': root,
                'temperature': 10.0,
                'layer': -2,
                'entropies': [],
                'varentropies': [],
                'threshold': (0.05, 0.05),
            }

            if use_prior_tree:
                prior_tree = tree_builder.build_from_dsus(traces)
                if verbose:
                    print(f"Current priors tree size: {prior_tree.count_subtree_size()}, height: {prior_tree.count_subtree_height()}")
            else:
                prior_tree = None

            for attempt in range(max_attempts):
                task.attempts = attempt + 1

                # Set target runs for each worker
                for rank, (model, tokenizer) in enumerate(workers):
                    context['dsu'] = dsu.to(model.device)
                    context['prior_tree'] = prior_tree
                    p = threading.Thread(
                        target=worker_closure,
                        args=(model, tokenizer, task, rank, context, batch_size, queue, verbose, return_trajectory),
                        daemon=True
                    )
                    p.start()
                    processes.append(p)

                entropies, varentropies = [], []

                results = [queue.get() for _ in processes]
                for p in processes:
                    p.join()

                for worker_results in results:
                    for b, r in enumerate(worker_results):
                        if isinstance(r, str):
                          print(f"Worker error: {r}")
                          continue # Error on the worker's side

                        stat = r['stats']
                        solution = r['completion']
                        if stat['score'] > stats['score']:
                            best_solution = solution

                        worker_attempt = batch_size * attempt + b
                        if 'trajectory' in r and stat['score'] == 1.0 and worker_attempt != 0:
                          trajectory = r['trajectory']
                          residual_collecton.trajectories.append(trajectory)

                        stats = merge_stats(stats, stat)
                        solutions.append(solution)

                        trajectory = r['changes']
                        nodes = []
                        for (activation, _) in trajectory:
                            node = dsu[activation]['index']
                            nodes.append(node)

                        for i, (_, action) in enumerate(trajectory[:-1]):
                            dsu.node_store[nodes[i]].add_child(nodes[i + 1], action)

                        Node.backpropagate([dsu.node_store[n] for n in nodes], stat['score'])

                        entropy_data = r['entropy_data']
                        worker_entropies, worker_varentropies = entropy_data

                        worker_entropies = worker_entropies[-(10000 // (len(processes) * batch_size)):]
                        entropies.extend(worker_entropies)

                        worker_varentropies = worker_varentropies[-(10000 // (len(processes) * batch_size)):]
                        varentropies.extend(worker_varentropies)

                entropy_threshold, varentropy_threshold = context['threshold']
                target_percentage = min(np.power(attempt, 1/3), 100)

                entropies = sorted(entropies, reverse=True)
                entropy_target_hits = max(0, int(max(len(entropies), 1000) * target_percentage / 100)-1)
                new_entropy_threshold = min(entropies[min(entropy_target_hits, max(len(entropies)-1, 0))], entropy_threshold)

                varentropies = sorted(varentropies, reverse=True)
                varentropy_target_hits = max(0, int(max(len(varentropies), 1000) * target_percentage / 100)-1)
                new_varentropy_threshold = min(varentropies[min(varentropy_target_hits, max(len(varentropies)-1, 0))], varentropy_threshold)

                context['threshold'] = (new_entropy_threshold, new_varentropy_threshold)

                processes = []
                gc.collect()

                if stats['correct']:
                    break

            task.stats = stats
            task.conversation.append(best_solution)
            finished_states.append(task)

            if len(residual_collecton.trajectories) > 0:
                residuals.append(residual_collecton)
            traces.append(dsu)

            progress_bar.update(1)
            gc.collect()
            update_pbar_stats(progress_bar, finished_states)
        except Exception as e:
            print(f"Error while running benchmark: {e}")
            print(f"Traceback:")
            traceback.print_exc()
            break

        gc.collect()

    print("Benchmark finished successfully")
    print("Dataset queue finished successfully")

    progress_bar.close()
    statistics = compile_statistics(finished_states)
    results = BenchmarkResults(items=finished_states, metadata=metadata)
    return statistics, results, residuals

In [27]:
from pydantic import TypeAdapter
adapter = TypeAdapter(list[ResidualCollection])

In [30]:
max_attempts = 8

metadata = {
    "model": MODEL_NAME,
    "attempts": max_attempts,
    **generation_config.__dict__
}

statistics, results, residuals = benchmark(
    coding_tasks[:5],
    workers,
    metadata,
    max_attempts=max_attempts,
    max_tasks=None,
    verbose=False,
    batch_size=1,
)

file_path_full = "Llama-3.2-3B.json"
results_dict = asdict(results)

with open(file_path_full, 'w') as f:
    json.dump(results_dict, f, indent=4)

print(statistics[0]['accuracy'], statistics[0]['avg score'])

residuals_path = "Llama-3.2-3B-residuals.json"
with open(residuals_path, 'w') as f:
    json_data = adapter.dump_json(residuals, indent=4)
    f.write(json_data.decode("utf-8"))

100%|██████████| 5/5 [07:47<00:00, 93.44s/it, acc=0.80, avg_score=0.90] 

Benchmark finished successfully
Dataset queue finished successfully
0.8 0.9


It doesn't show `acc=1.00`? Please, increase `max_attempts`.

In [31]:
with open(residuals_path, 'r') as file:
    json_data = file.read()
    residuals = adapter.validate_json(json_data)

You can also save residuals to finetune the model or to save the priors for another task.